<a href="https://colab.research.google.com/github/ai7532656-hash/colab/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import requests, zipfile, io

files_url = "https://ideami.com/llm_train"

response = requests.get(files_url)
zipfile.ZipFile(io.BytesIO(response.content)).extractall(".")

!pip install -r requirements.txt

In [29]:
import os, sys
import ipdb
from tqdm import tqdm
from datetime import datetime
import platform, shutil

import torch
import torch.nn as nn
from torch.nn import functional as F

#tokenizer
import sentencepiece as spm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.cuda.empty_cache()

In [30]:
# Architecture parameters

batch_size = 128
context = 512
embed_size = 384
n_layers = 7
n_heads = 7
BIAS = True

In [31]:
# Hyperparameters

lr = 3e-4
dropout = 0.05
weight_decay = 0.01
grad_clip = 1.0

In [32]:
# Training parameters

train_iters = 100000
eval_interval = 50
eval_iters = 10
compile = False
checkpoint_dir = 'models/'
checkpoint_fn ='latest.pt'
checkpoint_load_fn = 'latest.pt'
dtype = torch.bfloat16

# Mode
inference = False

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device: ", device)

Device:  cuda


In [33]:
# Logging

wandb_log = True
wandb_project = "llm1"
wandb_run_name = "llm1-" + datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

if wandb_log:
  import wandb
  wandb.init(project = wandb_project, name = wandb_run_name)

In [34]:
with open('wiki.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  print(text[10000:10300])

 that was used to represent a team in an old TV show, The A-Team. A capital a is written "A". Use a capital A at the start of a sentence if writing.

A is also a musical note, sometimes referred to as "La".

The letter 'A' was in the Phoenician alphabet's aleph. This symbol came from a simple pictur


In [35]:
# Tokenizer

sp = spm.SentencePieceProcessor(model_file='wiki_tokenizer.model')
vocab_size = sp.get_piece_size()
print(f"Tokenizer vocab size: {vocab_size}")

Tokenizer vocab size: 4096


In [36]:
encode = lambda s: sp.encode(s)
decode = lambda l: sp.decode(l)

print(encode("Once upon a time!"))

[612, 370, 698, 265, 261, 684, 36]


In [37]:
if os.path.exists("encoded_data.pt"):
  print("Loading encoding!")
  data = torch.load('encoded_data.pt')
else:
  data = torch.tensor(encode(text), dtype = torch.long)
  torch.save(data, "encoded_data.pt")

Loading encoding!


In [38]:
data_size = len(data)
spl = int(0.9 * data_size)
train_data = data[:spl]
val_data = data[spl:]

print("Training data: " + str(data_size))

Training data: 59211077


In [39]:
def get_batch(split):
  data = train_data if split == "train" else val_data
  inds = torch.randint(len(data)-context, (batch_size,))
  x = torch.stack([data[i: i+context] for i in inds])
  y = torch.stack([data[i+1: i+context+1] for i in inds])
  x,y = x.to(device), y.to(device)
  return x,y

x,y = get_batch("train")
print(x.shape, y.shape)
print(x[0][:10])
print(y[0][:10])

torch.Size([128, 512]) torch.Size([128, 512])
tensor([  13,   13, 4075,  621,  299,  261,  670,  278, 3195,  278],
       device='cuda:0')
tensor([  13, 4075,  621,  299,  261,  670,  278, 3195,  278,  264],
       device='cuda:0')


In [40]:
class GPT(nn.Module):
  def __init__(self):
       super().__init__()
       self.embeddings = nn.Embedding(vocab_size,embed_size) # Create embedding layer
       self.positions = nn.Embedding(context, embed_size) # Create basic positioning embeddings
       self.blocks = nn.Sequential(*[Block(n_heads) for _ in range(n_layers)]) # setup transformer blocks
       self.ln = nn.LayerNorm(embed_size) # normalization layers
       self.final_linear_layer = nn.Linear(embed_size, vocab_size, bias=BIAS) # feedforward linear layer
       self.apply(self._init_weights) # Initialize the weights

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)



  def forward(self, input, targets=None):
    loss = None
    BS, SL = input.shape
    emb = self.embeddings(input)
    pos = self.positions(torch.arange(SL, device=device))
    x = emb + pos
    x = self.blocks(x)
    x = self.ln(x)
    logits = self.final_linear_layer(x)

    if targets is not None:
      BS,SL, VS = logits.shape
      logits = logits.view(BS*SL, VS)
      targets = targets.view(BS*SL)
      loss = F.cross_entropy(logits, targets)

      #Manual calculation

      counts = logits.exp()
      prob = counts / counts.sum(-1, keepdim = True)
      loss2 = - prob[torch.arange(BS*SL), targets].log().mean()
    return logits, loss



  def generate(self, input, max=500):
       # SL = Sequence Length or context length
       for _ in range(max): # until you reach the maximum number of tokens
           input = input[:,-context:] #(1, input length until max of SL)
           logits, _ = self(input)  # (1, input length, 4096)
           logits = logits[:,-1,:]  # Pick last probability discarding the dimension (1, 4096)
           probs = F.softmax(logits, dim=-1) # (1,4096)
           next = torch.multinomial(probs, num_samples=1) # Sample next token value
           input = torch.cat((input,next),dim=1) # Add new token to the input
       return input

In [41]:
class Block(nn.Module):
  def __init__(self, n_heads):
    super().__init__()
    head_size = embed_size // n_heads
    self.ma = Multihead(n_heads, head_size)
    self.feed_forward = ForwardLayer(embed_size)
    self.ln1 = nn.LayerNorm(embed_size)
    self.ln2 = nn.LayerNorm(embed_size)

  def forward(self, x):
    x = x + self.ma(self.ln1(x))
    x = x + self.feed_forward(self.ln2(x))
    return x

In [42]:
class ForwardLayer(nn.Module):
  def __init__(self, embed_size):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(embed_size, 6*embed_size, bias=BIAS),
        nn.GELU(),
        nn.Linear(6*embed_size, embed_size, bias=BIAS),
        nn.Dropout(dropout)
    )

  def forward(self, x):
    x = self.network(x)
    return x

In [43]:
class Multihead(nn.Module):
  def __init__(self, n_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
    self.combine = nn.Linear(head_size * n_heads, embed_size, bias = BIAS)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    x = torch.cat([head(x) for head in self.heads], dim = -1)
    x = self.combine(x)
    x = self.dropout(x)
    return x

In [44]:
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.queries = nn.Linear(embed_size, head_size, bias=BIAS)
    self.keys = nn.Linear(embed_size, head_size, bias=BIAS)
    self.values = nn.Linear(embed_size, head_size, bias=BIAS)

    self.register_buffer('trail', torch.tril(torch.ones(context, context)))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    BS, SL, VS = x.shape
    q = self.queries(x)
    k = self.keys(x)
    v = self.values(x)

    attn_w = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
    attn_w = attn_w.masked_fill(self.tril[:SL,:SL]==0, float('-inf'))
    attn_w = F.softmax(attn_w, dim = -1)

    x = attn_w @ v

    return x

In [45]:
'''
x,y = get_batch("train")

model = GPT()
model = model.to(dtype)
model = model.to(device)



@torch.no_grad()
def generate_sample(input):
    t1 = torch.tensor(encode(input), dtype=torch.long, device=device) # Tokenize string -> (tensor of ids)
    t1 = t1[None,:]  # (1 , [size of ids])
    newgen = model.generate(t1,max=64)[0].tolist() # call the generate method, limit output size
    result=decode(newgen) # decode the result with the tokenizer to get back characters
    print(f"{result}")

generate_sample("The mountain in my city is") # Generate a sample
'''

'\nx,y = get_batch("train")\n\nmodel = GPT()\nmodel = model.to(dtype)\nmodel = model.to(device)\n\n\n\n@torch.no_grad()\ndef generate_sample(input):\n    t1 = torch.tensor(encode(input), dtype=torch.long, device=device) # Tokenize string -> (tensor of ids)\n    t1 = t1[None,:]  # (1 , [size of ids])\n    newgen = model.generate(t1,max=64)[0].tolist() # call the generate method, limit output size\n    result=decode(newgen) # decode the result with the tokenizer to get back characters\n    print(f"{result}")\n\ngenerate_sample("The mountain in my city is") # Generate a sample\n'

In [46]:
model = GPT()
model = model.to(dtype)
model = model.to(device)

if compile:
  print("Torch: Compiling the model!")
  model = torch.compile(model)

print(sum(p.numel() for p in model.parameters()) / 1e6, "Milion parameters")

19.837954 Milion parameters


In [47]:
@torch.no_grad()
def calculate_loss():
  out = {}
  model.eval()
  for split in ['train', 'eval']:
    l = torch.zeros(eval_iters)
    for i in range(eval_iters):
      x,y = get_batch(split)
      _, loss = model(x,y)
      l[i] = loss
    out[split]=l.mean().item()
  model.train()
  return out

print(calculate_loss())

TypeError: 'builtin_function_or_method' object is not subscriptable